# Qwen Variants Analysis

This notebook inspects BigGen Bench runs for Qwen 2.5-14B across paraphrase counts (`N`) and temperature sweeps. It computes summary statistics, fits a power-law curve to the MAD vs `N` trend, and highlights examples with the highest and lowest variance.


In [ ]:
from pathlib import Path
from collections import OrderedDict
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from IPython import get_ipython

NOTEBOOK_ROOT = Path.cwd().resolve()
_candidate_roots = [NOTEBOOK_ROOT, NOTEBOOK_ROOT.parent]
for _root in _candidate_roots:
    analysis_dir = _root / "analysis"
    if analysis_dir.exists() and str(_root) not in sys.path:
        sys.path.insert(0, str(_root))

from analysis import (
    collect_variant_statistics,
    load_variant_records,
    fit_power_law,
    power_law_predictions,
    summarize_temperature_runs,
    identify_variance_extremes,
)

get_ipython().run_line_magic('matplotlib', 'inline')
plt.style.use('seaborn-v0_8')


In [ ]:
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'results').exists():
    for parent in REPO_ROOT.parents:
        if (parent / 'results').exists():
            REPO_ROOT = parent
            break
RESULTS_DIR = REPO_ROOT / 'results' / 'biggen_bench'

N_RUNS = OrderedDict([
    (0, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p2.original.jsonl'),
    (2, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p2.jsonl'),
    (3, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p3.jsonl'),
    (5, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p5.jsonl'),
    (10, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p10.jsonl'),
    (20, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-p20.jsonl'),
])

TEMP_RUNS = OrderedDict([
    (0.0, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-temp0p0.jsonl'),
    (0.3, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-temp0p3.jsonl'),
    (0.7, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-temp0p7.jsonl'),
    (1.0, RESULTS_DIR / 'biggen_bench_pointwise_vanilla.base_pointwise.qwen2.5-14b-instruct-temp1p0.jsonl'),
])

available_n = OrderedDict((n, path) for n, path in N_RUNS.items() if path.exists())
missing_n = {n: str(path) for n, path in N_RUNS.items() if not path.exists()}
if missing_n:
    print('Warning: missing N-run results:')
    for n, path in missing_n.items():
        print(f"  N={n}: {path}")

available_temp = OrderedDict((t, path) for t, path in TEMP_RUNS.items() if path.exists())
missing_temp = {t: str(path) for t, path in TEMP_RUNS.items() if not path.exists()}
if missing_temp:
    print('Warning: missing temperature results:')
    for t, path in missing_temp.items():
        print(f"  temp={t}: {path}")


In [ ]:
n_stats = collect_variant_statistics({f'N={n}': path for n, path in available_n.items()})
if not n_stats.empty:
    if 'label' in n_stats.columns:
        n_stats['N'] = n_stats['label'].str.extract(r'N=(.*)').astype(float)
    elif 'result_path' in n_stats.columns:
        path_to_n = {str(path): float(n) for n, path in available_n.items()}
        n_stats['N'] = n_stats['result_path'].map(path_to_n)
    if 'N' in n_stats.columns:
        n_stats.sort_values('N', inplace=True)
else:
    print('No N-based statistics available.')
n_stats


In [ ]:
if 'N' not in n_stats.columns:
    raise RuntimeError('No N column available; ensure N-based runs were loaded above.')
fit_df = n_stats[n_stats['N'] > 0].dropna(subset=['mean_MAD_O']).copy()
if fit_df.empty:
    raise RuntimeError('Need at least one N>0 run with perturbations to fit power law.')
a_mad, b_mad = fit_power_law(fit_df['N'], fit_df['mean_MAD_O'])
print(f'Power-law fit (mean_MAD_O): y = {a_mad:.4f} * x^{b_mad:.4f}')
n_stats['mad_fit'] = power_law_predictions(n_stats['N'], a_mad, b_mad)
display_cols = [col for col in ['label', 'N', 'mean_MAD_O', 'mad_fit'] if col in n_stats.columns]
n_stats[display_cols]


In [ ]:
if not n_stats.empty and 'mad_fit' in n_stats.columns:
    fig, ax = plt.subplots(figsize=(6, 4))
    fit_df = n_stats[n_stats['N'] > 0].dropna(subset=['mean_MAD_O'])
    if not fit_df.empty:
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.scatter(fit_df['N'], fit_df['mean_MAD_O'], color='tab:blue', label='Observed')
        x_line = np.linspace(fit_df['N'].min(), fit_df['N'].max(), 200)
        ax.plot(x_line, power_law_predictions(x_line, a_mad, b_mad), color='tab:orange', label='Power-law fit')
        ax.set_xlabel('N (number of paraphrases)')
        ax.set_ylabel('Mean MAD$_{O}$')
        ax.set_title('Power-law behaviour of MAD vs N')
        ax.legend()
        ax.grid(True, which='both', ls='--', alpha=0.4)
        plt.show()


In [ ]:
temp_stats = summarize_temperature_runs(available_temp) if available_temp else pd.DataFrame()
temp_stats


In [ ]:
comparison_cols = ['mean_MAD_O', 'mean_RMSD_O', 'mean_score_std']
print('N-based runs:')
n_display_cols = [col for col in ['label'] + comparison_cols if col in n_stats.columns]
if n_display_cols:
    display(n_stats[n_display_cols])
else:
    print('No aggregated N statistics available.')
if not temp_stats.empty:
    print('\nTemperature runs:')
    temp_display_cols = [col for col in ['label'] + comparison_cols if col in temp_stats.columns]
    if temp_display_cols:
        display(temp_stats[temp_display_cols])
    else:
        print('Temperature statistics missing expected columns.')
else:
    print('No temperature statistics available.')


In [ ]:
if 20 in available_n:
    n20_records = load_variant_records(available_n[20])
    low_var, high_var = identify_variance_extremes(n20_records, top_k=3)
    print('Lowest-variance examples (N=20):')
    display(low_var[['example_id', 'variance', 'std', 'num_variants']])
    print('\nHighest-variance examples (N=20):')
    display(high_var[['example_id', 'variance', 'std', 'num_variants']])
else:
    print('N=20 results not available.')


In [ ]:
def inspect_example(record_df: pd.DataFrame, example_id: str) -> None:
    matches = record_df.loc[record_df['example_id'] == example_id]
    if matches.empty:
        print(f'Example {example_id} not found in records.')
        return
    row = matches.iloc[0]
    print(f"Example {example_id} | variance={row['variance']:.4f} | std={row['std']:.4f}")
    sections = row.get('sections') or {}
    assignment = sections.get('assignment')
    if assignment:
        print('\nAssignment:\n', assignment)
    student_answer = row.get('student_answer')
    if student_answer:
        print('\nStudent answer:\n', student_answer)
    variants = pd.Series(row.get('variant_scores', {}), dtype=float)
    if not variants.empty:
        display(variants.sort_values())

if 20 in available_n:
    n20_records = load_variant_records(available_n[20])
    low_var, high_var = identify_variance_extremes(n20_records, top_k=3)
    if not high_var.empty:
        for example_id in high_var['example_id']:
            print('\n=== High-variance example ===')
            inspect_example(n20_records, example_id)
    if not low_var.empty:
        for example_id in low_var['example_id']:
            print('\n=== Low-variance example ===')
            inspect_example(n20_records, example_id)
else:
    print('N=20 results not available.')
